In [ ]:
%cd ../../

In [ ]:
import sys
import math
from pathlib import Path

import yaml
import numpy as np
import torch
from loguru import logger
from transformers import BertModel
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from numpy import ndarray


In [ ]:
logger.remove()
logger.add(sys.stderr, level="DEBUG")

# Load things

In [ ]:
path = Path("src/gen_retrieval/configs.yaml")

with open(path) as file:
    conf = yaml.safe_load(file)

conf

In [ ]:
MODEL_NAME = conf['MODEL_DOCID']

model = BertModel.from_pretrained(MODEL_NAME).to(device="mps")

In [ ]:
path = Path(conf['INTERIM']['docid']['corpus'])

loaded = np.load(path, allow_pickle=True)['arr_0'].item()

input_ids = torch.from_numpy(loaded['input_ids']).to('mps')
attention_mask = torch.from_numpy(loaded['attention_mask']).to('mps')

In [ ]:
bsz = 10

class Embds(Dataset):
    def __init__(self, input_ids: ndarray, attention_mask: ndarray):
        super().__init__()

        self.input_ids = input_ids
        self.attention_mask = attention_mask

    def __len__(self) -> int:
        return len(self.input_ids)
    
    def __getitem__(self, index):
        return {
            'input_ids': input_ids[index],
            'attention_mask': attention_mask[index],
        }

dataset = Embds(input_ids, attention_mask)
loader = DataLoader(dataset, batch_size=bsz)

In [ ]:
total = math.ceil(len(dataset) / bsz)

embds_list = []
with tqdm(total=total) as pbar:
    for batch in loader:
        out = model(**batch).last_hidden_state.mean(dim=1).cpu().detach()
        embds_list.append(out)

        pbar.update(1)

        break

embds = torch.stack(embds_list)

In [ ]:
path = Path(conf['INTERIM']['docid']['embds'])
path.parent.mkdir(exist_ok=True, parents=True)

torch.save(embds, path)